## VLE 데이터 체크(1차 전처리)

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# 노트북에서 실행할 경우 프로젝트 최상위 폴더로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

STUDENT_VLE_PATH = PROJECT_ROOT / "data" / "raw" / "studentVle.csv"
VLE_PATH = PROJECT_ROOT / "data" / "raw" / "vle.csv"

print("프로젝트 위치:", PROJECT_ROOT)
print("studentVle 존재:", STUDENT_VLE_PATH.exists())
print("vle 존재:", VLE_PATH.exists())

프로젝트 위치: /Users/kyungduck/Desktop/SKN_AI/SKAI 2차 프로젝트
studentVle 존재: True
vle 존재: True


In [2]:
student_vle_sample = pd.read_csv(
    STUDENT_VLE_PATH,
    nrows=10_000
)

vle = pd.read_csv(VLE_PATH)

print("studentVle 샘플 크기:", student_vle_sample.shape)
print("vle 전체 크기:", vle.shape)

display(student_vle_sample.head())
display(vle.head())

studentVle 샘플 크기: (10000, 6)
vle 전체 크기: (6364, 6)


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


,"d""id_site""",code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,?,?
1,546712,AAA,2013J,oucontent,?,?
2,546998,AAA,2013J,resource,?,?
3,546888,AAA,2013J,url,?,?
4,547035,AAA,2013J,resource,?,?


In [3]:
print("===== studentVle 샘플 정보 =====")
student_vle_sample.info()

print("\nstudentVle 결측치")
display(student_vle_sample.isnull().sum())

print("\n===== vle 정보 =====")
vle.info()

print("\nvle 결측치")
display(vle.isnull().sum())

===== studentVle 샘플 정보 =====
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   code_module        10000 non-null  str  
 1   code_presentation  10000 non-null  str  
 2   id_student         10000 non-null  int64
 3   id_site            10000 non-null  int64
 4   date               10000 non-null  int64
 5   sum_click          10000 non-null  int64
dtypes: int64(4), str(2)
memory usage: 547.0 KB

studentVle 결측치


code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64


===== vle 정보 =====
<class 'pandas.DataFrame'>
RangeIndex: 6364 entries, 0 to 6363
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   d"id_site"         6364 non-null   int64
 1   code_module        6364 non-null   str  
 2   code_presentation  6364 non-null   str  
 3   activity_type      6364 non-null   str  
 4   week_from          6364 non-null   str  
 5   week_to            6364 non-null   str  
dtypes: int64(1), str(5)
memory usage: 407.2 KB

vle 결측치


d"id_site"           0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
dtype: int64

In [4]:
# 정확한 컬럼명 확인
print("vle 컬럼명:", vle.columns.tolist())

# 주차 컬럼에 들어 있는 실제 값 확인
print("\nweek_from 값 예시:")
print(vle["week_from"].value_counts(dropna=False).head(15))

print("\nweek_to 값 예시:")
print(vle["week_to"].value_counts(dropna=False).head(15))

vle 컬럼명: ['d"id_site"', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']

week_from 값 예시:
week_from
?     5243
18      91
1       84
28      63
9       52
21      51
27      49
11      47
15      46
29      45
20      43
2       41
10      40
22      38
8       36
Name: count, dtype: int64

week_to 값 예시:
week_to
?     5243
18      91
1       80
28      63
9       54
21      51
27      49
11      47
15      46
2       45
29      45
20      43
10      40
22      38
8       34
Name: count, dtype: int64


In [5]:
# 전처리용 복사본 생성
vle_clean = vle.copy()

# 잘못된 컬럼명 수정
vle_clean = vle_clean.rename(
    columns={'d"id_site"': "id_site"}
)

# '?'를 결측치로 바꾸고 정수형으로 변환
for col in ["week_from", "week_to"]:
    vle_clean[col] = pd.to_numeric(
        vle_clean[col].replace("?", pd.NA),
        errors="coerce"
    ).astype("Int64")

print("수정된 컬럼명:", vle_clean.columns.tolist())
print("\n자료형:")
print(vle_clean.dtypes)

print("\n결측치:")
print(vle_clean.isna().sum())

수정된 컬럼명: ['id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']

자료형:
id_site              int64
code_module            str
code_presentation      str
activity_type          str
week_from            Int64
week_to              Int64
dtype: object

결측치:
id_site                 0
code_module             0
code_presentation       0
activity_type           0
week_from            5243
week_to              5243
dtype: int64


In [6]:
VLE_KEYS = [
    "code_module",
    "code_presentation",
    "id_site"
]

# 핵심 컬럼 결측치 확인
print("핵심 컬럼 결측치:")
print(
    vle_clean[
        VLE_KEYS + ["activity_type"]
    ].isna().sum()
)

# 병합 키 중복 확인
duplicate_count = vle_clean.duplicated(
    subset=VLE_KEYS,
    keep=False
).sum()

print("\n병합 키 중복 행 수:", duplicate_count)
print("activity_type 종류 수:", vle_clean["activity_type"].nunique())

print("\nactivity_type별 콘텐츠 수:")
display(
    vle_clean["activity_type"]
    .value_counts()
    .rename_axis("activity_type")
    .reset_index(name="content_count")
)

핵심 컬럼 결측치:
code_module          0
code_presentation    0
id_site              0
activity_type        0
dtype: int64

병합 키 중복 행 수: 0
activity_type 종류 수: 20

activity_type별 콘텐츠 수:


,activity_type,content_count
0,resource,2660
1,subpage,1055
2,oucontent,996
3,url,886
4,forumng,194
5,quiz,127
6,page,102
7,oucollaborate,82
8,questionnaire,61
9,ouwiki,49


In [7]:
MERGE_KEYS = [
    "code_module",
    "code_presentation",
    "id_site"
]

student_vle_sample_merged = student_vle_sample.merge(
    vle_clean[
        MERGE_KEYS + ["activity_type"]
    ],
    on=MERGE_KEYS,
    how="left",
    validate="many_to_one",
    indicator=True
)

print("병합 결과:")
print(student_vle_sample_merged["_merge"].value_counts())

print(
    "\nactivity_type이 연결되지 않은 행:",
    student_vle_sample_merged["activity_type"].isna().sum()
)

display(student_vle_sample_merged.head())

병합 결과:
_merge
both          10000
left_only         0
right_only        0
Name: count, dtype: int64

activity_type이 연결되지 않은 행: 0


,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type,_merge
0,AAA,2013J,28400,546652,-10,4,forumng,both
1,AAA,2013J,28400,546652,-10,1,forumng,both
2,AAA,2013J,28400,546652,-10,1,forumng,both
3,AAA,2013J,28400,546614,-10,11,homepage,both
4,AAA,2013J,28400,546714,-10,1,oucontent,both


In [8]:
CHUNK_SIZE = 500_000

total_rows = 0
date_min = None
date_max = None
click_min = None
click_max = None

before_course_rows = 0
nonpositive_click_rows = 0
missing_counts = None

for chunk_number, chunk in enumerate(
    pd.read_csv(
        STUDENT_VLE_PATH,
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows += len(chunk)

    # 결측치 누적
    chunk_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = chunk_missing
    else:
        missing_counts = missing_counts.add(
            chunk_missing,
            fill_value=0
        )

    # 날짜 범위
    current_date_min = chunk["date"].min()
    current_date_max = chunk["date"].max()

    date_min = (
        current_date_min
        if date_min is None
        else min(date_min, current_date_min)
    )
    date_max = (
        current_date_max
        if date_max is None
        else max(date_max, current_date_max)
    )

    # 클릭 수 범위
    current_click_min = chunk["sum_click"].min()
    current_click_max = chunk["sum_click"].max()

    click_min = (
        current_click_min
        if click_min is None
        else min(click_min, current_click_min)
    )
    click_max = (
        current_click_max
        if click_max is None
        else max(click_max, current_click_max)
    )

    # 개강 전 활동과 비정상 클릭 확인
    before_course_rows += (chunk["date"] < 0).sum()
    nonpositive_click_rows += (chunk["sum_click"] <= 0).sum()

    if chunk_number % 5 == 0:
        print(f"{chunk_number}개 청크 확인 완료")

print("\n===== 전체 점검 결과 =====")
print("전체 행 수:", f"{total_rows:,}")
print("활동 날짜 범위:", date_min, "~", date_max)
print("클릭 수 범위:", click_min, "~", click_max)
print("개강 전 활동 행 수:", f"{before_course_rows:,}")
print("0 이하 클릭 행 수:", f"{nonpositive_click_rows:,}")

print("\n컬럼별 결측치:")
print(missing_counts.astype(int))

5개 청크 확인 완료
10개 청크 확인 완료
15개 청크 확인 완료
20개 청크 확인 완료

===== 전체 점검 결과 =====
전체 행 수: 10,655,280
활동 날짜 범위: -25 ~ 269
클릭 수 범위: 1 ~ 6977
개강 전 활동 행 수: 688,988
0 이하 클릭 행 수: 0

컬럼별 결측치:
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64


In [9]:
student_vle_sample_weekly = student_vle_sample_merged.copy()

# 개강 전 여부
student_vle_sample_weekly["is_pre_course"] = (
    student_vle_sample_weekly["date"] < 0
).astype(int)

# 개강 후 데이터에만 주차 부여
student_vle_sample_weekly["week_index"] = pd.NA

after_start_mask = student_vle_sample_weekly["date"] >= 0

student_vle_sample_weekly.loc[
    after_start_mask,
    "week_index"
] = (
    student_vle_sample_weekly.loc[
        after_start_mask,
        "date"
    ] // 7
) + 1

student_vle_sample_weekly["week_index"] = (
    student_vle_sample_weekly["week_index"]
    .astype("Int64")
)

display(
    student_vle_sample_weekly[
        [
            "date",
            "week_index",
            "is_pre_course",
            "sum_click",
            "activity_type"
        ]
    ].head(20)
)

print("개강 전 샘플 행:", student_vle_sample_weekly["is_pre_course"].sum())
print("샘플 주차 범위:", student_vle_sample_weekly["week_index"].min(),
      "~", student_vle_sample_weekly["week_index"].max())

,date,week_index,is_pre_course,sum_click,activity_type
0,-10,<NA>,1,4,forumng
1,-10,<NA>,1,1,forumng
2,-10,<NA>,1,1,forumng
3,-10,<NA>,1,11,homepage
4,-10,<NA>,1,1,oucontent
5,-10,<NA>,1,8,forumng
6,-10,<NA>,1,2,subpage
7,-10,<NA>,1,15,oucontent
8,-10,<NA>,1,17,oucontent
9,-10,<NA>,1,1,url


개강 전 샘플 행: 10000
샘플 주차 범위: <NA> ~ <NA>


In [10]:
sample_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        STUDENT_VLE_PATH,
        chunksize=500_000
    ),
    start=1
):
    chunk_sample = chunk.sample(
        frac=0.001,
        random_state=42 + chunk_number
    )
    sample_parts.append(chunk_sample)

student_vle_random_sample = pd.concat(
    sample_parts,
    ignore_index=True
)

# VLE 콘텐츠 정보 연결
student_vle_random_sample = student_vle_random_sample.merge(
    vle_clean[
        MERGE_KEYS + ["activity_type"]
    ],
    on=MERGE_KEYS,
    how="left",
    validate="many_to_one"
)

# 개강 전 여부
student_vle_random_sample["is_pre_course"] = (
    student_vle_random_sample["date"] < 0
).astype(int)

# 개강 후 주차 계산
student_vle_random_sample["week_index"] = pd.NA

after_start_mask = student_vle_random_sample["date"] >= 0

student_vle_random_sample.loc[
    after_start_mask,
    "week_index"
] = (
    student_vle_random_sample.loc[
        after_start_mask,
        "date"
    ] // 7
) + 1

student_vle_random_sample["week_index"] = (
    student_vle_random_sample["week_index"]
    .astype("Int64")
)

print("대표 샘플 행 수:", len(student_vle_random_sample))
print(
    "날짜 범위:",
    student_vle_random_sample["date"].min(),
    "~",
    student_vle_random_sample["date"].max()
)
print(
    "주차 범위:",
    student_vle_random_sample["week_index"].min(),
    "~",
    student_vle_random_sample["week_index"].max()
)
print(
    "개강 전 샘플 행:",
    student_vle_random_sample["is_pre_course"].sum()
)

display(
    student_vle_random_sample[
        [
            "date",
            "week_index",
            "is_pre_course",
            "sum_click",
            "activity_type"
        ]
    ].sample(20, random_state=42)
)

대표 샘플 행 수: 10655
날짜 범위: -25 ~ 269
주차 범위: 1 ~ 39
개강 전 샘플 행: 721


,date,week_index,is_pre_course,sum_click,activity_type
1350,63,10,0,1,forumng
2012,136,20,0,1,quiz
1965,189,28,0,3,forumng
9359,-17,<NA>,1,1,url
251,101,15,0,1,forumng
33,94,14,0,2,forumng
6569,86,13,0,2,subpage
8610,100,15,0,4,homepage
4792,122,18,0,3,forumng
265,16,3,0,8,homepage


In [11]:
# [
#     "code_module",        # 강좌
#     "code_presentation",  # 강좌 운영 회차
#     "id_student",         # 학생
#     "week_index"          # 7일 단위 주차
# ]

WEEKLY_KEYS = [
    "code_module",
    "code_presentation",
    "id_student",
    "week_index"
]

PRE_COURSE_KEYS = [
    "code_module",
    "code_presentation",
    "id_student"
]

weekly_parts = []
pre_course_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        STUDENT_VLE_PATH,
        chunksize=500_000
    ),
    start=1
):
    # 개강 전과 개강 후 분리
    pre_course = chunk[chunk["date"] < 0].copy()
    after_start = chunk[chunk["date"] >= 0].copy()

    # 개강 후 활동을 7일 단위로 변환
    after_start["week_index"] = (
        after_start["date"] // 7
    ) + 1

    # 학생·강좌·운영 회차·주차별 부분 집계
    weekly_part = (
        after_start
        .groupby(
            WEEKLY_KEYS,
            as_index=False,
            sort=False
        )
        .agg(
            total_clicks=("sum_click", "sum"),
            interaction_rows=("sum_click", "size")
        )
    )

    weekly_parts.append(weekly_part)

    # 개강 전 활동은 학생·강좌 단위로 별도 집계
    pre_course_part = (
        pre_course
        .groupby(
            PRE_COURSE_KEYS,
            as_index=False,
            sort=False
        )
        .agg(
            pre_course_clicks=("sum_click", "sum"),
            pre_course_interaction_rows=("sum_click", "size")
        )
    )

    pre_course_parts.append(pre_course_part)

    if chunk_number % 5 == 0:
        print(f"{chunk_number}개 청크 집계 완료")

# 청크 경계에서 나뉜 동일 학생 기록을 다시 합산
weekly_base = (
    pd.concat(weekly_parts, ignore_index=True)
    .groupby(
        WEEKLY_KEYS,
        as_index=False,
        sort=False
    )
    .agg(
        total_clicks=("total_clicks", "sum"),
        interaction_rows=("interaction_rows", "sum")
    )
)

pre_course_base = (
    pd.concat(pre_course_parts, ignore_index=True)
    .groupby(
        PRE_COURSE_KEYS,
        as_index=False,
        sort=False
    )
    .agg(
        pre_course_clicks=("pre_course_clicks", "sum"),
        pre_course_interaction_rows=(
            "pre_course_interaction_rows",
            "sum"
        )
    )
)

print("\n주차별 집계 행 수:", f"{len(weekly_base):,}")
print("개강 전 집계 행 수:", f"{len(pre_course_base):,}")

print(
    "주차별 키 중복:",
    weekly_base.duplicated(WEEKLY_KEYS).sum()
)

print(
    "집계에 포함된 전체 원본 행:",
    f"{weekly_base['interaction_rows'].sum() + pre_course_base['pre_course_interaction_rows'].sum():,}"
)

display(weekly_base.head())
display(pre_course_base.head())

5개 청크 집계 완료
10개 청크 집계 완료
15개 청크 집계 완료
20개 청크 집계 완료

주차별 집계 행 수: 579,438
개강 전 집계 행 수: 23,809
주차별 키 중복: 0
집계에 포함된 전체 원본 행: 10,655,280


,code_module,code_presentation,id_student,week_index,total_clicks,interaction_rows
0,AAA,2013J,345357,1,122,33
1,AAA,2013J,336207,1,144,30
2,AAA,2013J,331358,1,355,73
3,AAA,2013J,321942,1,64,39
4,AAA,2013J,324002,1,154,45


,code_module,code_presentation,id_student,pre_course_clicks,pre_course_interaction_rows
0,AAA,2013J,28400,215,55
1,AAA,2013J,30268,102,34
2,AAA,2013J,31604,169,38
3,AAA,2013J,32885,295,66
4,AAA,2013J,38053,277,56


In [12]:
weekly_day_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        STUDENT_VLE_PATH,
        usecols=[
            "code_module",
            "code_presentation",
            "id_student",
            "date"
        ],
        chunksize=500_000
    ),
    start=1
):
    # 개강 후 활동만 사용
    chunk = chunk[chunk["date"] >= 0].copy()

    # 7일 단위 주차 생성
    chunk["week_index"] = (
        chunk["date"] // 7
    ) + 1

    # 같은 학생이 같은 날짜에 여러 콘텐츠를 이용한 경우 하나로 정리
    unique_days_part = chunk[
        WEEKLY_KEYS + ["date"]
    ].drop_duplicates()

    weekly_day_parts.append(unique_days_part)

    if chunk_number % 5 == 0:
        print(f"{chunk_number}개 청크 활동일 확인 완료")

# 청크 사이에서 중복된 학생·날짜까지 최종 제거
unique_weekly_days = (
    pd.concat(
        weekly_day_parts,
        ignore_index=True
    )
    .drop_duplicates(
        subset=WEEKLY_KEYS + ["date"]
    )
)

# 학생·강좌·주차별 실제 활동일 수
weekly_active_days = (
    unique_weekly_days
    .groupby(
        WEEKLY_KEYS,
        as_index=False,
        sort=False
    )
    .agg(
        active_days=("date", "nunique")
    )
)

# 기존 주차 집계 결과에 연결
weekly_features = weekly_base.merge(
    weekly_active_days,
    on=WEEKLY_KEYS,
    how="left",
    validate="one_to_one"
)

print("active_days 결측치:", weekly_features["active_days"].isna().sum())
print(
    "active_days 범위:",
    weekly_features["active_days"].min(),
    "~",
    weekly_features["active_days"].max()
)
print(
    "병합 후 키 중복:",
    weekly_features.duplicated(WEEKLY_KEYS).sum()
)

display(weekly_features.head())

5개 청크 활동일 확인 완료
10개 청크 활동일 확인 완료
15개 청크 활동일 확인 완료
20개 청크 활동일 확인 완료
active_days 결측치: 0
active_days 범위: 1 ~ 7
병합 후 키 중복: 0


,code_module,code_presentation,id_student,week_index,total_clicks,interaction_rows,active_days
0,AAA,2013J,345357,1,122,33,5
1,AAA,2013J,336207,1,144,30,6
2,AAA,2013J,331358,1,355,73,6
3,AAA,2013J,321942,1,64,39,6
4,AAA,2013J,324002,1,154,45,6


In [13]:
import gc

# 이전 계산에서 사용한 큰 임시 객체 정리
del weekly_day_parts
del unique_weekly_days
del weekly_active_days
gc.collect()

0

In [17]:
weekly_site_parts = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        STUDENT_VLE_PATH,
        usecols=[
            "code_module",
            "code_presentation",
            "id_student",
            "id_site",
            "date"
        ],
        chunksize=500_000
    ),
    start=1
):
    # 개강 후 데이터
    chunk = chunk[chunk["date"] >= 0].copy()

    # 7일 단위 주차 생성
    chunk["week_index"] = (
        chunk["date"] // 7
    ) + 1

    # 같은 주차에 같은 콘텐츠를 여러 번 사용해도 하나로 계산
    unique_sites_part = chunk[
        WEEKLY_KEYS + ["id_site"]
    ].drop_duplicates()

    weekly_site_parts.append(unique_sites_part)

    if chunk_number % 5 == 0:
        print(f"{chunk_number}개 청크 콘텐츠 확인 완료")

# 청크 사이의 중복까지 제거
unique_weekly_sites = (
    pd.concat(
        weekly_site_parts,
        ignore_index=True
    )
    .drop_duplicates(
        subset=WEEKLY_KEYS + ["id_site"]
    )
)

# 학생·강좌·주차별 고유 콘텐츠 수
weekly_unique_sites = (
    unique_weekly_sites
    .groupby(
        WEEKLY_KEYS,
        as_index=False,
        sort=False
    )
    .agg(
        unique_sites=("id_site", "nunique")
    )
)

# 기존 Feature에 연결
weekly_features = weekly_features.merge(
    weekly_unique_sites,
    on=WEEKLY_KEYS,
    how="left",
    validate="one_to_one"
)

print(
    "unique_sites 결측치:",
    weekly_features["unique_sites"].isna().sum()
)
print(
    "unique_sites 범위:",
    weekly_features["unique_sites"].min(),
    "~",
    weekly_features["unique_sites"].max()
)
print(
    "병합 후 키 중복:",
    weekly_features.duplicated(WEEKLY_KEYS).sum()
)

display(weekly_features.head())

5개 청크 콘텐츠 확인 완료
10개 청크 콘텐츠 확인 완료
15개 청크 콘텐츠 확인 완료
20개 청크 콘텐츠 확인 완료
unique_sites 결측치: 0
unique_sites 범위: 1 ~ 268
병합 후 키 중복: 0


,code_module,code_presentation,id_student,week_index,total_clicks,interaction_rows,active_days,unique_sites_x,unique_sites_y,avg_clicks_per_active_day,unique_sites
0,AAA,2013J,345357,1,122,33,5,14,14,24.400000,14
1,AAA,2013J,336207,1,144,30,6,12,12,24.000000,12
2,AAA,2013J,331358,1,355,73,6,32,32,59.166667,32
3,AAA,2013J,321942,1,64,39,6,6,6,10.666667,6
4,AAA,2013J,324002,1,154,45,6,16,16,25.666667,16


In [18]:
# 활동일당 평균 클릭 수
weekly_features["avg_clicks_per_active_day"] = (
    weekly_features["total_clicks"]
    / weekly_features["active_days"]
)

# 콘텐츠당 평균 클릭 수
weekly_features["avg_clicks_per_site"] = (
    weekly_features["total_clicks"]
    / weekly_features["unique_sites"]
)

print(
    "추가 Feature 결측치:",
    weekly_features[
        [
            "avg_clicks_per_active_day",
            "avg_clicks_per_site"
        ]
    ].isna().sum()
)

display(
    weekly_features[
        WEEKLY_KEYS + [
            "total_clicks",
            "interaction_rows",
            "active_days",
            "unique_sites",
            "avg_clicks_per_active_day",
            "avg_clicks_per_site"
        ]
    ].head()
)

추가 Feature 결측치: avg_clicks_per_active_day    0
avg_clicks_per_site          0
dtype: int64


,code_module,code_presentation,id_student,week_index,total_clicks,interaction_rows,active_days,unique_sites,avg_clicks_per_active_day,avg_clicks_per_site
0,AAA,2013J,345357,1,122,33,5,14,24.400000,8.714286
1,AAA,2013J,336207,1,144,30,6,12,24.000000,12.000000
2,AAA,2013J,331358,1,355,73,6,32,59.166667,11.093750
3,AAA,2013J,321942,1,64,39,6,6,10.666667,10.666667
4,AAA,2013J,324002,1,154,45,6,16,25.666667,9.625000


In [19]:
import gc

del weekly_site_parts
del unique_weekly_sites
del weekly_unique_sites
gc.collect()

783

In [21]:
CORE_ACTIVITY_TYPES = [
    "forumng",
    "quiz",
    "oucontent",
    "resource"
]

TYPE_KEYS = WEEKLY_KEYS + ["activity_type"]
activity_type_parts = []
unmatched_rows = 0

vle_lookup = vle_clean[
    MERGE_KEYS + ["activity_type"]
].copy()

for chunk_number, chunk in enumerate(
    pd.read_csv(
        STUDENT_VLE_PATH,
        chunksize=500_000
    ),
    start=1
):
    # 개강 후 활동만 사용
    chunk = chunk[chunk["date"] >= 0].copy()

    # 7일 단위 주차 생성
    chunk["week_index"] = (
        chunk["date"] // 7
    ) + 1

    # 콘텐츠 ID를 이용해 활동 유형 연결
    chunk = chunk.merge(
        vle_lookup,
        on=MERGE_KEYS,
        how="left",
        validate="many_to_one"
    )

    unmatched_rows += chunk["activity_type"].isna().sum()

    # 학생·강좌·주차·활동 유형별 클릭 합계
    type_part = (
        chunk
        .groupby(
            TYPE_KEYS,
            as_index=False,
            sort=False
        )
        .agg(
            type_clicks=("sum_click", "sum")
        )
    )

    activity_type_parts.append(type_part)

    if chunk_number % 5 == 0:
        print(f"{chunk_number}개 청크 활동 유형 집계 완료")

# 청크 사이에서 나뉜 동일 그룹 재집계
activity_type_totals = (
    pd.concat(
        activity_type_parts,
        ignore_index=True
    )
    .groupby(
        TYPE_KEYS,
        as_index=False,
        sort=False
    )
    .agg(
        type_clicks=("type_clicks", "sum")
    )
)

# 한 주 동안 이용한 활동 유형 수
activity_type_count = (
    activity_type_totals
    .groupby(
        WEEKLY_KEYS,
        as_index=False,
        sort=False
    )
    .agg(
        activity_type_count=("activity_type", "nunique")
    )
)

# 핵심 4종을 제외한 활동은 other로 통합
activity_type_totals["activity_group"] = (
    activity_type_totals["activity_type"]
    .where(
        activity_type_totals["activity_type"]
        .isin(CORE_ACTIVITY_TYPES),
        "other"
    )
)

activity_group_totals = (
    activity_type_totals
    .groupby(
        WEEKLY_KEYS + ["activity_group"],
        as_index=False,
        sort=False
    )
    .agg(
        activity_clicks=("type_clicks", "sum")
    )
)

# 활동 유형을 열 형태로 변환
activity_wide = (
    activity_group_totals
    .pivot(
        index=WEEKLY_KEYS,
        columns="activity_group",
        values="activity_clicks"
    )
    .fillna(0)
    .reset_index()
)

activity_wide.columns.name = None

activity_wide = activity_wide.rename(
    columns={
        "forumng": "forumng_clicks",
        "quiz": "quiz_clicks",
        "oucontent": "oucontent_clicks",
        "resource": "resource_clicks",
        "other": "other_clicks"
    }
)

ACTIVITY_FEATURES = [
    "forumng_clicks",
    "quiz_clicks",
    "oucontent_clicks",
    "resource_clicks",
    "other_clicks"
]

# 일부 데이터에 특정 유형이 전혀 없어도 컬럼 생성
for col in ACTIVITY_FEATURES:
    if col not in activity_wide.columns:
        activity_wide[col] = 0

# 기존 Feature에 병합
weekly_features = (
    weekly_features
    .merge(
        activity_type_count,
        on=WEEKLY_KEYS,
        how="left",
        validate="one_to_one"
    )
    .merge(
        activity_wide[
            WEEKLY_KEYS + ACTIVITY_FEATURES
        ],
        on=WEEKLY_KEYS,
        how="left",
        validate="one_to_one"
    )
)

weekly_features[
    ["activity_type_count"] + ACTIVITY_FEATURES
] = weekly_features[
    ["activity_type_count"] + ACTIVITY_FEATURES
].fillna(0)

print("활동 유형 연결 실패 행:", unmatched_rows)
print(
    "클릭 합계 일치:",
    activity_type_totals["type_clicks"].sum()
    == weekly_features["total_clicks"].sum()
)
print(
    "병합 후 키 중복:",
    weekly_features.duplicated(WEEKLY_KEYS).sum()
)

display(weekly_features.head())

5개 청크 활동 유형 집계 완료
10개 청크 활동 유형 집계 완료
15개 청크 활동 유형 집계 완료
20개 청크 활동 유형 집계 완료
활동 유형 연결 실패 행: 0
클릭 합계 일치: True
병합 후 키 중복: 0


,code_module,code_presentation,id_student,week_index,total_clicks,interaction_rows,active_days,unique_sites_x,unique_sites_y,avg_clicks_per_active_day,unique_sites,avg_clicks_per_site,activity_type_count,forumng_clicks,quiz_clicks,oucontent_clicks,resource_clicks,other_clicks
0,AAA,2013J,345357,1,122,33,5,14,14,24.400000,14,8.714286,5,8.0,0.0,82.0,0.0,32.0
1,AAA,2013J,336207,1,144,30,6,12,12,24.000000,12,12.000000,4,66.0,0.0,49.0,0.0,29.0
2,AAA,2013J,331358,1,355,73,6,32,32,59.166667,32,11.093750,7,157.0,0.0,140.0,2.0,56.0
3,AAA,2013J,321942,1,64,39,6,6,6,10.666667,6,10.666667,2,53.0,0.0,0.0,0.0,11.0
4,AAA,2013J,324002,1,154,45,6,16,16,25.666667,16,9.625000,5,31.0,0.0,71.0,0.0,52.0


In [22]:
print("전체 행 수:", f"{len(weekly_features):,}")
print("전체 컬럼 수:", len(weekly_features.columns))

print("\n전체 컬럼:")
for number, column in enumerate(
    weekly_features.columns,
    start=1
):
    print(number, column)

suffix_columns = [
    col for col in weekly_features.columns
    if col.endswith("_x") or col.endswith("_y")
]

print("\n중복 실행 의심 컬럼:", suffix_columns)

activity_sum = weekly_features[
    ACTIVITY_FEATURES
].sum(axis=1)

print(
    "활동별 클릭 합계가 total_clicks와 다른 행:",
    (activity_sum != weekly_features["total_clicks"]).sum()
)

print(
    "전체 결측치 수:",
    weekly_features.isna().sum().sum()
)

print(
    "키 중복:",
    weekly_features.duplicated(WEEKLY_KEYS).sum()
)

전체 행 수: 579,438
전체 컬럼 수: 18

전체 컬럼:
1 code_module
2 code_presentation
3 id_student
4 week_index
5 total_clicks
6 interaction_rows
7 active_days
8 unique_sites_x
9 unique_sites_y
10 avg_clicks_per_active_day
11 unique_sites
12 avg_clicks_per_site
13 activity_type_count
14 forumng_clicks
15 quiz_clicks
16 oucontent_clicks
17 resource_clicks
18 other_clicks

중복 실행 의심 컬럼: ['unique_sites_x', 'unique_sites_y']
활동별 클릭 합계가 total_clicks와 다른 행: 0
전체 결측치 수: 0
키 중복: 0


In [25]:
# 정수형 Feature 정리
INTEGER_FEATURES = [
    "total_clicks",
    "interaction_rows",
    "active_days",
    "unique_sites",
    "activity_type_count",
    "forumng_clicks",
    "quiz_clicks",
    "oucontent_clicks",
    "resource_clicks",
    "other_clicks"
]

weekly_features[INTEGER_FEATURES] = (
    weekly_features[INTEGER_FEATURES]
    .astype("int64")
)

# 결과 정렬
weekly_features = (
    weekly_features
    .sort_values(WEEKLY_KEYS)
    .reset_index(drop=True)
)

pre_course_base = (
    pre_course_base
    .sort_values(PRE_COURSE_KEYS)
    .reset_index(drop=True)
)

# 저장 경로
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_OUTPUT_PATH = (
    INTERIM_DIR / "vle_weekly_features.csv"
)
PRE_COURSE_OUTPUT_PATH = (
    INTERIM_DIR / "vle_pre_course_features.csv"
)
VLE_CLEAN_OUTPUT_PATH = (
    INTERIM_DIR / "vle_metadata_clean.csv"
)

# CSV 저장
weekly_features.to_csv(
    WEEKLY_OUTPUT_PATH,
    index=False
)

pre_course_base.to_csv(
    PRE_COURSE_OUTPUT_PATH,
    index=False
)

vle_clean.to_csv(
    VLE_CLEAN_OUTPUT_PATH,
    index=False
)

print("저장 완료")
print(
    WEEKLY_OUTPUT_PATH.name,
    f"{WEEKLY_OUTPUT_PATH.stat().st_size / 1024**2:.1f} MB"
)
print(
    PRE_COURSE_OUTPUT_PATH.name,
    f"{PRE_COURSE_OUTPUT_PATH.stat().st_size / 1024**2:.1f} MB"
)
print(
    VLE_CLEAN_OUTPUT_PATH.name,
    f"{VLE_CLEAN_OUTPUT_PATH.stat().st_size / 1024**2:.1f} MB"
)

저장 완료
vle_weekly_features.csv 33.5 MB
vle_pre_course_features.csv 0.5 MB
vle_metadata_clean.csv 0.2 MB
